In [1]:
"""
Step 6: Model Training
======================
"""

import numpy as np
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
)
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import os

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# 1. LOAD DATA

print("\n1. LOADING TRAINING AND VALIDATION DATA")
print("-" * 80)

train_data = np.load(
    '../data/processed/train_data.npz',
    allow_pickle=True 
)

# Convert to float32 immediately to fix the object type issue
X_train = np.asarray(train_data['X'], dtype=np.float32)
y_train = np.asarray(train_data['y'], dtype=np.float32)

val_data = np.load(
    '../data/processed/val_data.npz',
    allow_pickle=True
)
X_val = np.asarray(val_data['X'], dtype=np.float32)
y_val = np.vstack(val_data['y']).astype(np.float32) 

assert X_train.dtype != object, "X_train contains object dtype!"
assert y_train.dtype != object, "y_train contains object dtype!"
assert X_val.dtype != object, "X_val contains object dtype!"
assert y_val.dtype != object, "y_val contains object dtype!"
assert y_train.ndim == 2, "y_train is not 2D!"
assert y_val.ndim == 2, "y_val is not 2D!"

print(f"✓ Training set: {X_train.shape[0]:,} samples")
print(f"✓ Validation set: {X_val.shape[0]:,} samples")
print(f"✓ Input shape: {X_train.shape[1:]}")
print(f"✓ Output shape: {y_train.shape[1:]}")


1. LOADING TRAINING AND VALIDATION DATA
--------------------------------------------------------------------------------
✓ Training set: 177,322 samples
✓ Validation set: 38,016 samples
✓ Input shape: (10, 12)
✓ Output shape: (2,)


In [3]:
# 2. DEFINE TRAINING CALLBACKS

print("\n2. SETTING UP TRAINING CALLBACKS")
print("-" * 80)

def get_callbacks(model_name):
    """Create callbacks for training"""
    
    # Early stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
    
    # Model checkpoint - save best model
    checkpoint = ModelCheckpoint(
        filepath=f'../data/models/{model_name}_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
    
    # Reduce learning rate when validation loss plateaus
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
    
    # TensorBoard for visualization
    tensorboard = TensorBoard(
        log_dir=f'logs/{model_name}_{datetime.now().strftime("%Y%m%d-%H%M%S")}',
        histogram_freq=1
    )
    
    return [early_stop, checkpoint, reduce_lr, tensorboard]

print("✓ Callbacks configured:")
print("  - Early Stopping (patience=15)")
print("  - Model Checkpoint (save best)")
print("  - Learning Rate Reduction (factor=0.5, patience=5)")
print("  - TensorBoard Logging")

# Must match the definition from Step 5 exactly
class AttentionLayer(keras.layers.Layer):
    """Custom attention layer"""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
    
    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

    def get_config(self):
        config = super(AttentionLayer, self).get_config()
        return config


2. SETTING UP TRAINING CALLBACKS
--------------------------------------------------------------------------------
✓ Callbacks configured:
  - Early Stopping (patience=15)
  - Model Checkpoint (save best)
  - Learning Rate Reduction (factor=0.5, patience=5)
  - TensorBoard Logging


In [4]:
# 3. TRAIN ALL MODELS

print("\n3. TRAINING ALL MODELS")
print("-" * 80)

model_names = [
    'LSTM',
    'GRU',
    'Bidirectional_LSTM',
    'CNN_LSTM',
    'Attention_LSTM'
]

# Training configuration
BATCH_SIZE = 32
EPOCHS = 100  

training_results = {}

for model_name in model_names:
    print("\n" + "=" * 80)
    print(f"TRAINING: {model_name}")
    print("=" * 80)
    
    # Define custom objects dictionary
    custom_obj = {'AttentionLayer': AttentionLayer} if model_name == 'Attention_LSTM' else None

    # Load model architecture
    model = keras.models.load_model(
        f'/Users/dilumsamarathunga/Projects/MachineLearning/child_growth_prediction/data/models/{model_name}_architecture.keras',
        custom_objects=custom_obj
    )
    
    print(f"Recompiling {model_name} with gradient clipping (clipnorm=1.0)...")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
        loss='mse',
        metrics=['mae', keras.metrics.RootMeanSquaredError(name='rmse')]
    )
    
    # Get callbacks
    callbacks_list = get_callbacks(model_name)
    
    # Train model
    print(f"\nStarting training for {model_name}...")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Max epochs: {EPOCHS}")
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks_list,
        verbose=1
    )
    
    # Store results
    training_results[model_name] = {
        'history': history.history,
        'best_epoch': np.argmin(history.history['val_loss']) + 1,
        'best_val_loss': np.min(history.history['val_loss']),
        'best_val_mae': history.history['val_mae'][np.argmin(history.history['val_loss'])],
        'best_val_rmse': history.history['val_rmse'][np.argmin(history.history['val_loss'])],
        'total_epochs': len(history.history['loss'])
    }
    
    print(f"\n✓ {model_name} training completed!")
    print(f"  Best epoch: {training_results[model_name]['best_epoch']}")
    print(f"  Best val_loss: {training_results[model_name]['best_val_loss']:.4f}")
    print(f"  Best val_mae: {training_results[model_name]['best_val_mae']:.4f}")
    print(f"  Best val_rmse: {training_results[model_name]['best_val_rmse']:.4f}")
    print(f"  Total epochs run: {training_results[model_name]['total_epochs']}")


3. TRAINING ALL MODELS
--------------------------------------------------------------------------------

TRAINING: LSTM
Recompiling LSTM with gradient clipping (clipnorm=1.0)...

Starting training for LSTM...
Batch size: 32
Max epochs: 100
Epoch 1/100


/Users/dilumsamarathunga/Desktop/child_growth_prediction_01/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 26 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5536/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 221.5267 - mae: 7.5915 - rmse: 13.1927
Epoch 1: val_loss improved from None to 45.55027, saving model to ../data/models/LSTM_best.keras

Epoch 1: finished saving model to ../data/models/LSTM_best.keras
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 48s 8ms/step - loss: 81.9807 - mae: 5.1713 - rmse: 9.0543 - val_loss: 45.5503 - val_mae: 4.5832 - val_rmse: 6.7491 - learning_rate: 0.0010
Epoch 2/100
5538/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 40.0905 - mae: 3.9902 - rmse: 6.3310
Epoch 2: val_loss did not improve from 45.55027
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 47s 9ms/step - loss: 38.2938 - mae: 3.8640 - rmse: 6.1882 - val_loss: 82.6272 - val_mae: 6.5004 - val_rmse: 9.0899 - learning_rate: 0.0010
Epoch 3/100
5535/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 33.4933 - mae: 3.5534 - rmse: 5.7870
Epoch 3: val_loss did not improve from 45.55027
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 45s 8ms/step - loss: 33.1631 - mae: 3.5205 - rmse: 5.7587 - val_loss: 81.9305 - 

/Users/dilumsamarathunga/Desktop/child_growth_prediction_01/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5542/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 244.4982 - mae: 7.3227 - rmse: 13.6290
Epoch 1: val_loss improved from None to 30.84428, saving model to ../data/models/Bidirectional_LSTM_best.keras

Epoch 1: finished saving model to ../data/models/Bidirectional_LSTM_best.keras
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 83.8196 - mae: 5.0136 - rmse: 9.1553 - val_loss: 30.8443 - val_mae: 3.4663 - val_rmse: 5.5538 - learning_rate: 0.0010
Epoch 2/100
5536/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 40.3374 - mae: 4.0044 - rmse: 6.3498
Epoch 2: val_loss did not improve from 30.84428
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 48s 9ms/step - loss: 38.1326 - mae: 3.8559 - rmse: 6.1752 - val_loss: 32.2055 - val_mae: 3.5605 - val_rmse: 5.6750 - learning_rate: 0.0010
Epoch 3/100
5540/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 32.5562 - mae: 3.4665 - rmse: 5.7052
Epoch 3: val_loss did not improve from 30.84428
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 47s 8ms/step - loss: 31.8995 - mae: 3.3890 - rmse: 5

/Users/dilumsamarathunga/Desktop/child_growth_prediction_01/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 28 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/100
5526/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 214.8533 - mae: 6.9559 - rmse: 12.8112
Epoch 1: val_loss improved from None to 29.15968, saving model to ../data/models/CNN_LSTM_best.keras

Epoch 1: finished saving model to ../data/models/CNN_LSTM_best.keras
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 78.0734 - mae: 4.9613 - rmse: 8.8359 - val_loss: 29.1597 - val_mae: 3.2665 - val_rmse: 5.4000 - learning_rate: 0.0010
Epoch 2/100
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 42.5924 - mae: 4.1411 - rmse: 6.5255
Epoch 2: val_loss improved from 29.15968 to 27.62993, saving model to ../data/models/CNN_LSTM_best.keras

Epoch 2: finished saving model to ../data/models/CNN_LSTM_best.keras
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 40.8647 - mae: 4.0337 - rmse: 6.3926 - val_loss: 27.6299 - val_mae: 3.1104 - val_rmse: 5.2564 - learning_rate: 0.0010
Epoch 3/100
5533/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 36.3807 - mae: 3.7736 - rmse: 6.0314
Epoch 3: val

/Users/dilumsamarathunga/Desktop/child_growth_prediction_01/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 30 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5537/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 299.3187 - mae: 9.2484 - rmse: 15.6745
Epoch 1: val_loss improved from None to 70.01952, saving model to ../data/models/Attention_LSTM_best.keras

Epoch 1: finished saving model to ../data/models/Attention_LSTM_best.keras
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 51s 9ms/step - loss: 136.8626 - mae: 7.2140 - rmse: 11.6988 - val_loss: 70.0195 - val_mae: 5.7853 - val_rmse: 8.3678 - learning_rate: 0.0010
Epoch 2/100
5539/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 49.5278 - mae: 4.4547 - rmse: 7.0259
Epoch 2: val_loss did not improve from 70.01952
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 52s 9ms/step - loss: 43.2474 - mae: 4.0999 - rmse: 6.5763 - val_loss: 95.5833 - val_mae: 7.0494 - val_rmse: 9.7767 - learning_rate: 0.0010
Epoch 3/100
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 36.4119 - mae: 3.6936 - rmse: 6.0336
Epoch 3: val_loss did not improve from 70.01952
5542/5542 ━━━━━━━━━━━━━━━━━━━━ 50s 9ms/step - loss: 35.3487 - mae: 3.6210 - rmse: 5.9455 

In [5]:
# 4. SAVE TRAINING RESULTS

print("\n4. SAVING TRAINING RESULTS")
print("-" * 80)

with open('../data/models/training_results.pkl', 'wb') as f:
    pickle.dump(training_results, f)

print("✓ Saved training results to: ../data/models/training_results.pkl")


4. SAVING TRAINING RESULTS
--------------------------------------------------------------------------------
✓ Saved training results to: ../data/models/training_results.pkl


In [6]:
# 5. COMPARE MODEL PERFORMANCE

print("\n5. MODEL PERFORMANCE COMPARISON")
print("-" * 80)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': model_names,
    'Best_Epoch': [training_results[m]['best_epoch'] for m in model_names],
    'Val_Loss': [training_results[m]['best_val_loss'] for m in model_names],
    'Val_MAE': [training_results[m]['best_val_mae'] for m in model_names],
    'Val_RMSE': [training_results[m]['best_val_rmse'] for m in model_names],
    'Total_Epochs': [training_results[m]['total_epochs'] for m in model_names]
})

# Sort by validation loss
comparison_df = comparison_df.sort_values('Val_Loss')

print("\nModel Performance Ranking:")
print(comparison_df.to_string(index=False))

comparison_df.to_csv('../data/models/model_comparison.csv', index=False)
print("\n✓ Saved comparison to: ..data/models/model_comparison.csv")

# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
print(f"\n BEST MODEL: {best_model_name}")
print(f"   Val Loss: {comparison_df.iloc[0]['Val_Loss']:.4f}")
print(f"   Val MAE: {comparison_df.iloc[0]['Val_MAE']:.4f}")
print(f"   Val RMSE: {comparison_df.iloc[0]['Val_RMSE']:.4f}")


5. MODEL PERFORMANCE COMPARISON
--------------------------------------------------------------------------------

Model Performance Ranking:
             Model  Best_Epoch  Val_Loss  Val_MAE  Val_RMSE  Total_Epochs
          CNN_LSTM           8 26.651552 2.985242  5.162514            23
Bidirectional_LSTM           1 30.844280 3.466348  5.553763            16
              LSTM           1 45.550270 4.583248  6.749094            16
               GRU           1 47.047577 4.608500  6.859124            16
    Attention_LSTM           1 70.019516 5.785285  8.367766            16

✓ Saved comparison to: ..data/models/model_comparison.csv

 BEST MODEL: CNN_LSTM
   Val Loss: 26.6516
   Val MAE: 2.9852
   Val RMSE: 5.1625


In [7]:
# 6. PLOT TRAINING HISTORY

print("\n6. GENERATING TRAINING PLOTS")
print("-" * 80)

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, model_name in enumerate(model_names):
    history = training_results[model_name]['history']
    
    ax = axes[idx]
    
    ax.plot(history['loss'], label='Train Loss', linewidth=2)
    ax.plot(history['val_loss'], label='Val Loss', linewidth=2)
    
    best_epoch = training_results[model_name]['best_epoch'] - 1
    ax.axvline(x=best_epoch, color='red', linestyle='--', 
               label=f'Best Epoch ({best_epoch+1})')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'{model_name}\nBest Val Loss: {training_results[model_name]["best_val_loss"]:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

if len(model_names) < 6:
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('../data/models/training_history.png', dpi=300, bbox_inches='tight')
print("✓ Saved training plots to: ../data/models/training_history.png")
plt.close()


6. GENERATING TRAINING PLOTS
--------------------------------------------------------------------------------
✓ Saved training plots to: ../data/models/training_history.png


In [8]:
# 7. PLOT METRICS COMPARISON

print("\n7. GENERATING METRICS COMPARISON PLOT")
print("-" * 80)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Val_Loss', 'Val_MAE', 'Val_RMSE']
titles = ['Validation Loss (MSE)', 'Validation MAE', 'Validation RMSE']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    
    sorted_df = comparison_df.sort_values(metric)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric])
    bars[0].set_color('green')
    
    ax.set_xlabel(title)
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='x')
    
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', 
                ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/models/metrics_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved metrics comparison to: ../data/models/metrics_comparison.png")
plt.close()


7. GENERATING METRICS COMPARISON PLOT
--------------------------------------------------------------------------------
✓ Saved metrics comparison to: ../data/models/metrics_comparison.png


In [9]:
# 8. SAVE BEST MODEL INFO

print("\n8. SAVING BEST MODEL INFORMATION")
print("-" * 80)

best_model_info = {
    'best_model_name': best_model_name,
    'best_model_path': f'../data/models/{best_model_name}_best.keras',
    'val_loss': float(comparison_df.iloc[0]['Val_Loss']),
    'val_mae': float(comparison_df.iloc[0]['Val_MAE']),
    'val_rmse': float(comparison_df.iloc[0]['Val_RMSE']),
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

with open('../data/models/best_model_info.pkl', 'wb') as f:
    pickle.dump(best_model_info, f)

print("✓ Saved best model info to: ../data/models/best_model_info.pkl")


8. SAVING BEST MODEL INFORMATION
--------------------------------------------------------------------------------
✓ Saved best model info to: ../data/models/best_model_info.pkl


In [10]:
# 9. TRAINING SUMMARY

print("\n" + "=" * 80)
print("TRAINING SUMMARY")
print("=" * 80)

print(f"\n✓ Trained {len(model_names)} models successfully")
print(f"✓ Best performing model: {best_model_name}")

print("\nPerformance Metrics (Validation Set):")
print(f"  Loss (MSE): {best_model_info['val_loss']:.4f}")
print(f"  MAE: {best_model_info['val_mae']:.4f}")
print(f"  RMSE: {best_model_info['val_rmse']:.4f}")

print("\nInterpretation:")
print(f"  - On average, predictions are off by ±{best_model_info['val_mae']:.2f} units")
print(f"  - For height: ~±{best_model_info['val_mae']:.2f} cm")
print(f"  - For weight: ~±{best_model_info['val_mae']:.2f} kg")

print("\nSaved Files:")
print("  - Best models: ../data/models/*_best.keras")
print("  - Training results: ../data/models/training_results.pkl")
print("  - Model comparison: ../data/models/model_comparison.csv")
print("  - Training plots: data/models/training_history.png")
print("  - Metrics comparison: ../data/models/metrics_comparison.png")


TRAINING SUMMARY

✓ Trained 5 models successfully
✓ Best performing model: CNN_LSTM

Performance Metrics (Validation Set):
  Loss (MSE): 26.6516
  MAE: 2.9852
  RMSE: 5.1625

Interpretation:
  - On average, predictions are off by ±2.99 units
  - For height: ~±2.99 cm
  - For weight: ~±2.99 kg

Saved Files:
  - Best models: ../data/models/*_best.keras
  - Training results: ../data/models/training_results.pkl
  - Model comparison: ../data/models/model_comparison.csv
  - Training plots: data/models/training_history.png
  - Metrics comparison: ../data/models/metrics_comparison.png
